# Create lithium supply scenarios using *minerals_supply_scenarios*

This notebook creates the lithium supply scenarios based on S&P database. The following scenarios are created:
- Baseline
- Ambitious
- Very Ambitious

The development of the supply scenarios is facilitated by the *minerals_supply_scenarios* Python library, designed to assist in the development of scenarios for minerals supply based on asset-level data. The tool streamlines the process of importing, processing, and analyzing mining asset data from the S&P Capital IQ Pro database. By default, supply scenarios are created considering the development stage of the mining projects, thus reflecting different levels of production expansion. Scenario data is exported as a scenario data file that is integrated into *premise*.

Get the tool from: https://github.com/robyistrate/minerals_supply_scenarios

In [1]:
# Add minerals_supply_scenarios to path and import
import sys
import os
ROOT_DIR = os.path.abspath(os.path.join("../../minerals_supply_scenarios"))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

import minerals_supply_scenarios as mss

In [2]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import datetime

SCENARIO_DATA_PATH = Path("../scenario_data")
RESULTS_PATH = Path("../results")

## Create scenario data

In [10]:
SP_DATASET_PATH = SCENARIO_DATA_PATH / "external" / "SPGlobal_Export_10-27-2025_075e226a-b1a1-44a0-adc8-71be16274415.xls"
scenario_timeframe = (2020,2035,1)
list_of_years = [str(year) for year in range(scenario_timeframe[0], scenario_timeframe[1]+1, scenario_timeframe[2])]

lithium_scenarios = mss.MetalSupplyScenarios(
    commodity="Lithium",
    dataset_path=SP_DATASET_PATH,
    timeframe=scenario_timeframe,
    specifics={"Deposit Type": ["Brine (Salar)"]},
    exclude={"Country": ["Canada"], 
             "Deposit Type": ["Brine (Salar), Pegmatite Hosted"]},
    export_dir=SCENARIO_DATA_PATH / "external"
    )

Considering only: {'Deposit Type': ['Brine (Salar)']}
Excluding: {'Country': ['Canada'], 'Deposit Type': ['Brine (Salar), Pegmatite Hosted']}
****************************************
Creating supply scenarios for lithium
Importing raw dataset from S&P database...
Applying strategy: fill_data_gaps
Applying strategy: estimate_future_production
Applying strategy: create_scenarios_data
Applying strategy: harmonize_production_data
Exporting premise scenario data file
*****************************
Processing report
Number projects in updated dataset : 145
Number projects with production in updated dataset : Production 2020     8
Production 2021     8
Production 2022     9
Production 2023    12
Production 2024    19
Production 2025    22
Production 2026    23
Production 2027    27
Production 2028    29
Production 2029    29
Production 2030    34
Production 2031    34
Production 2032    34
Production 2033    34
Production 2034    34
Production 2035    34
dtype: int64
Number projects in scenari

c:\Users\istrateir\OneDrive - Universiteit Leiden\Research\minerals_supply_scenarios\minerals_supply_scenarios\supply_scenarios.py:261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sc_data[production_columns] = sc_data[production_columns].fillna(0)
c:\Users\istrateir\OneDrive - Universiteit Leiden\Research\minerals_supply_scenarios\minerals_supply_scenarios\supply_scenarios.py:261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sc_data[production_columns] = sc_data[production_columns].fillna(0)
c:\Users\istrateir\OneDrive - Universiteit Leiden\Research\m

In [11]:
# Explore discarded projects due to lack of production capacity
all_projects = list(set(lithium_scenarios.sp_dataset_updated.index))
considered_projects = list(set(lithium_scenarios.scenario_data["Project ID"]))
excluded_projects = list(set(all_projects) - set(considered_projects))

print("Number of projects discarded:", len(all_projects) - len(considered_projects))
print("% of projects discarded:", round(len(excluded_projects) *100 / len(all_projects)), "%")

Number of projects discarded: 111
% of projects discarded: 77 %


In [12]:
lithium_scenarios.scenario_data["Project Name"].values

array(['Cauchari-Olaroz', 'Chaerhan Lake', 'Cuenca Centenario-Ratones',
       'East Taijinair', 'Sal de los Angeles', 'Salar de Atacama',
       'Salar de Atacama', 'Salar de Olaroz', 'Salar del Hombre Muerto',
       'Silver Peak', 'Zhabuye', 'Cauchari-Olaroz', 'Chaerhan Lake',
       'Cuenca Centenario-Ratones', 'East Taijinair', 'Fort Cady',
       'Kachi', 'Lakkor Tso Salt Lake', 'Mariana', 'Maricunga', 'Paradox',
       'Pastos Grandes', 'Qinghai Yiliping', 'Sal de los Angeles',
       'Sal de Vida', 'Salar de Atacama', 'Salar de Atacama',
       'Salar de Olaroz', 'Salar del Hombre Muerto', 'Salar del Rincon',
       'Silver Peak', 'Tres Quebradas', 'Uyuni Salt Flat', 'Vulcan',
       'Zhabuye', 'Cauchari-Olaroz', 'Chaerhan Lake',
       'Cuenca Centenario-Ratones', 'East Taijinair', 'Fort Cady',
       'Hombre Muerto North', 'Kachi', 'Kuska', 'Laguna Verde',
       'Lakkor Tso Salt Lake', 'Mariana', 'Maricunga', 'Paradox',
       'Pastos Grandes', 'PPG', 'Qinghai Yiliping', 'Sa

In [13]:
# Calculate % of projects excluded by development stage
dev_stage_discarded_projects = lithium_scenarios.sp_dataset_updated.loc[excluded_projects].groupby(
    'Development Stage').size().sort_values(ascending=False).round(0).reset_index(name='Number')
dev_stage_discarded_projects["Share"] = dev_stage_discarded_projects["Number"].divide(len(excluded_projects)).multiply(100).round(0)

dev_stage_discarded_projects

,Development Stage,Number,Share
0,Target Outline,46,41.0
1,Exploration,29,26.0
2,Reserves Development,14,13.0
3,Grassroots,9,8.0
4,Prefeas/Scoping,7,6.0
5,Limited Production,2,2.0
6,Advanced Exploration,1,1.0
7,Commissioning,1,1.0
8,Preproduction,1,1.0
9,Satellite,1,1.0


## Create premise scenario data file based on available LCI datasets

Projects for which LCI datasets are not available are excluded

In [14]:
lithium_scenario_data = lithium_scenarios.premise_scenario_data.copy()

In [15]:
lithium_scenario_data[lithium_scenario_data["variables"]=="Production|Lithium|Brine (Salar)|Lakkor Tso Salt Lake"]

,scenario,region,variables,unit,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
6,S&P-Ambitious,CN,Production|Lithium|Brine (Salar)|Lakkor Tso Sa...,kt/year,0.0,0.0,0.0,1.386,11.07855,14.7714,18.46425,18.46425,18.46425,29.55225,40.64025,46.18425,46.18425,46.18425,46.18425,46.18425
39,S&P-Baseline,CN,Production|Lithium|Brine (Salar)|Lakkor Tso Sa...,kt/year,0.0,0.0,0.0,0.000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
72,S&P-Very Ambitious,CN,Production|Lithium|Brine (Salar)|Lakkor Tso Sa...,kt/year,0.0,0.0,0.0,1.386,11.07855,14.7714,18.46425,18.46425,18.46425,29.55225,40.64025,46.18425,46.18425,46.18425,46.18425,46.18425


In [17]:
# Import the map between project names and LCI dataset
map_lithium_projects_to_lci = pd.read_excel(SCENARIO_DATA_PATH / "lithium_projects_list.xlsx")
map_lithium_projects_to_lci.head()

,Project name,Name in S&P,Country,Location,Development stage,Status,Technology,LCI dataset,Li concentration,Comments
0,Salar de Cauchari-Olaroz,Cauchari-Olaroz,AR,AR,Operating,Operating,Chemical-based,df_rotary_dryer_Salar de Cauchari-Olaroz,0.000500,NaN
1,Chaerhan,Chaerhan Lake,CN,CN-NWG,Operating,Operating,DLE,df_rotary_dryer_Chaerhan,0.000662,NaN
2,Salar de Centenario,Cuenca Centenario-Ratones,AR,AR,Preproduction,Operating,DLE,df_rotary_dryer_Salar de Centenario,0.000309,Production started Diciembre 2024: https://www...
3,East Taijinar,East Taijinair,CN,CN-NWG,Operating,Operating,DLE,df_rotary_dryer_East Taijinar,0.000800,https://rslithium.com/research-report-on-lithi...
4,Sal de los Angeles,Sal de los Angeles,AR,AR,Preproduction,Announced,DLE,df_rotary_dryer_Sal de los Angeles,0.000193,NaN


In [18]:
# Check projects for which there is not LCI (they aren't in the mapping file)
print("Projects not in the mapping file:")
print("******************")
for proj in list(set(lithium_scenario_data["variables"])):
    project_name = proj.split('|')[-1]

    if project_name not in list(map_lithium_projects_to_lci["Name in S&P"]):
        print(project_name)

Projects not in the mapping file:
******************
Laguna Verde
Fort Cady
Sal de Oro
Salta Lithium
Mariana
Viento Andino
Salar de Cauchari
Kuska
Paradox


In [19]:
# Check that all projects with LCI are in the scenario
print("Projects not in the scenario:")
print("******************")
for project_name in list(map_lithium_projects_to_lci["Name in S&P"]):
    if project_name not in [i.split('|')[-1] for i in list(set(lithium_scenario_data["variables"]))]:
        print(project_name)

Projects not in the scenario:
******************


In [20]:
lithium_scenarios_premise_adjusted = []

for index, row in lithium_scenario_data.iterrows():
    project_var = row["variables"]
    project_name = project_var.split('|')[-1]
    try:
        project_lci = map_lithium_projects_to_lci[
            map_lithium_projects_to_lci["Name in S&P"] == project_name]["LCI dataset"].values[0]        
        lithium_scenarios_premise_adjusted.append(
                [row["scenario"], row["region"], row["variables"], row["unit"]] + [row[year] for year in list_of_years])
    except IndexError:
        pass
lithium_scenarios_premise_adjusted = pd.DataFrame(lithium_scenarios_premise_adjusted, columns = lithium_scenario_data.columns)

In [21]:
# Export scenario data file for use in premise
lithium_scenarios_premise_adjusted[["scenario", "region", "variables", "unit", "2020", "2025", "2030", "2035"]].to_csv(
    SCENARIO_DATA_PATH / "external" / f"lithium_scenario_data_with_LCIs_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv", index=False)

In [22]:
# Statistics projects excluded due to lack of LCI datasets
all_projects_with_production = list(set(lithium_scenarios.premise_scenario_data["variables"]))
considered_projects_with_LCIs = list(set(lithium_scenarios_premise_adjusted["variables"]))
excluded_projects = list(set(all_projects_with_production) - set(considered_projects_with_LCIs))

print("Number of projects dwith production:", len(all_projects_with_production))
print("Number of projects discarded due to lack of LCIs:", len(excluded_projects))
print("% of projects discarded:", round(len(excluded_projects) *100 / len(all_projects)), "%")
print("Final number of projects:", len(considered_projects_with_LCIs))

Number of projects dwith production: 33
Number of projects discarded due to lack of LCIs: 9
% of projects discarded: 6 %
Final number of projects: 24


## Generate scenario results

Results for Figure 3 in the main manuscript, including production per country, project, and total

In [23]:
# First, annonymize project names. Each project name is assigned a
# unique number (except for names containing "Other projects," which should be assigned as "Others")
projects_id_path = SCENARIO_DATA_PATH / "external" / "anonymized_projects_id.yaml"

# Check if the file already exists
if projects_id_path.exists():
    # If the file exists, open and load the existing content
    print("Opnening existing annonymized projects ID list")
    with open(projects_id_path, "r") as yaml_file:
        project_ids = yaml.safe_load(yaml_file)
else:
    # If the file does not exist, create a new mapping
    print("Creating the annonymized projects ID list")
    project_ids = {}

    counter = 1
    for proj in list(set(lithium_scenarios_premise_adjusted["variables"])):
        project_ids[proj] = f"#{counter}"
        counter += 1

    # Save the mapping as a new YAML file
    with open(projects_id_path, "w") as yaml_file:
        yaml.dump(project_ids, yaml_file, default_flow_style=False)

Opnening existing annonymized projects ID list


In [24]:
# Production by country in each scenario
lithium_production_by_country = lithium_scenarios_premise_adjusted[["scenario", "region"] + [year for year in list_of_years]].groupby(["scenario", "region"]).sum()

# Total production in each scenario
total_lithium_production = lithium_production_by_country.groupby(level='scenario').sum()

# Production shares by project in each scenario
production_by_project = lithium_scenarios_premise_adjusted[["scenario", "variables"] + [year for year in list_of_years]].groupby(
            ["scenario", "variables"]).sum().reset_index().rename(columns={"variables": "project"})
production_by_project['project'] = production_by_project['project'].replace(project_ids)
production_by_project.set_index(['scenario', 'project'], inplace=True)
sums_by_scenario_project = production_by_project.groupby('scenario').sum()
production_share_by_project = production_by_project.div(sums_by_scenario_project)

# Production by technology
production_by_technology = []
for index, row in lithium_scenarios_premise_adjusted.iterrows():
    project_name = row["variables"].split('|')[-1]
    project_technology = map_lithium_projects_to_lci[map_lithium_projects_to_lci["Name in S&P"] == project_name]["Technology"].values
    production_entry = (row["scenario"], project_technology[0]) + tuple(row[list_of_years])
    production_by_technology.append(production_entry)

production_by_technology_df = pd.DataFrame(production_by_technology, columns=["scenario", "technology"] + list_of_years)
production_by_technology_df = production_by_technology_df.groupby(['scenario', 'technology']).sum()

In [25]:
# Calculate total litihum production including excluded projects
total_lithium_production_in_SP = lithium_scenario_data.groupby(["scenario"]).sum().drop(columns=["variables", "unit", "region"])

In [26]:
# Export results
lithium_production_by_country.to_csv(RESULTS_PATH / f"lithium_production_by_country_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv")
total_lithium_production.to_csv(RESULTS_PATH / f"lithium_production_total_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv")
total_lithium_production_in_SP.to_csv(RESULTS_PATH / f"lithium_production_total_SP_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv")
production_share_by_project.to_csv(RESULTS_PATH / f"lithium_production_share_by_project_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv")
production_by_technology_df.to_csv(RESULTS_PATH / f"lithium_production_by_technology_{datetime.datetime.today().strftime('%d-%m-%Y')}.csv")